# UK Electricity Demand Analysis

## 1. Setup and Data Loading

This notebook analyses historical electricity demand in Great Britain using the 2024 Historic Demand Data published by the National Energy System Operator (NESO).

The analysis explores temporal patterns in National Demand (ND), including monthly and seasonal variation, weekday and weekend differences, typical daily demand profiles, and the relationship between electricity demand and embedded renewable generation.

### Analysis Workflow

The notebook follows these main stages:

1. Import the required Python libraries.
2. Load and inspect the NESO electricity demand dataset.
3. Assess data quality and prepare the data for analysis.
4. Engineer time-based features for temporal analysis.
5. Explore electricity demand patterns through summary statistics and visualisations.
6. Investigate the relationship between electricity demand and embedded renewable generation.

In [1]:
import pandas as pd

### Load the Dataset

The 2024 NESO Historic Demand Data is loaded from the project's raw data directory into a pandas DataFrame for analysis.

In [2]:
df = pd.read_csv("../data/raw/demanddata_2024.csv")

## 2. Initial Data Exploration

The dataset is inspected to understand its structure, dimensions, variables, data types, and overall data quality before proceeding with cleaning and analysis.

In [3]:
df.head()

,SETTLEMENT_DATE,SETTLEMENT_PERIOD,ND,TSD,ENGLAND_WALES_DEMAND,EMBEDDED_WIND_GENERATION,EMBEDDED_WIND_CAPACITY,EMBEDDED_SOLAR_GENERATION,EMBEDDED_SOLAR_CAPACITY,NON_BM_STOR,...,IFA_FLOW,IFA2_FLOW,BRITNED_FLOW,MOYLE_FLOW,EAST_WEST_FLOW,NEMO_FLOW,NSL_FLOW,ELECLINK_FLOW,VIKING_FLOW,GREENLINK_FLOW
0,01-JAN-2024,1,21783,23466,19539,2804,6488,0,16793,0,...,417,-4,666,-179,-92,905,62,998,572,0
1,01-JAN-2024,2,22521,24103,20286,2834,6488,0,16793,0,...,430,-4,661,-60,-8,901,-15,998,575,0
2,01-JAN-2024,3,22194,24754,20070,2868,6488,0,16793,0,...,834,-4,661,-118,-1,807,-396,997,422,0
3,01-JAN-2024,4,21510,24505,19424,2901,6488,0,16793,0,...,839,-4,661,-270,-84,803,-473,997,401,0
4,01-JAN-2024,5,20619,23977,18674,2933,6488,0,16793,0,...,53,-5,661,-340,-103,537,-856,998,28,0


In [4]:
df.shape

(17568, 22)

In [5]:
df.columns.tolist()

['SETTLEMENT_DATE',
 'SETTLEMENT_PERIOD',
 'ND',
 'TSD',
 'ENGLAND_WALES_DEMAND',
 'EMBEDDED_WIND_GENERATION',
 'EMBEDDED_WIND_CAPACITY',
 'EMBEDDED_SOLAR_GENERATION',
 'EMBEDDED_SOLAR_CAPACITY',
 'NON_BM_STOR',
 'PUMP_STORAGE_PUMPING',
 'SCOTTISH_TRANSFER',
 'IFA_FLOW',
 'IFA2_FLOW',
 'BRITNED_FLOW',
 'MOYLE_FLOW',
 'EAST_WEST_FLOW',
 'NEMO_FLOW',
 'NSL_FLOW',
 'ELECLINK_FLOW',
 'VIKING_FLOW',
 'GREENLINK_FLOW']

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17568 entries, 0 to 17567
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   SETTLEMENT_DATE            17568 non-null  str  
 1   SETTLEMENT_PERIOD          17568 non-null  int64
 2   ND                         17568 non-null  int64
 3   TSD                        17568 non-null  int64
 4   ENGLAND_WALES_DEMAND       17568 non-null  int64
 5   EMBEDDED_WIND_GENERATION   17568 non-null  int64
 6   EMBEDDED_WIND_CAPACITY     17568 non-null  int64
 7   EMBEDDED_SOLAR_GENERATION  17568 non-null  int64
 8   EMBEDDED_SOLAR_CAPACITY    17568 non-null  int64
 9   NON_BM_STOR                17568 non-null  int64
 10  PUMP_STORAGE_PUMPING       17568 non-null  int64
 11  SCOTTISH_TRANSFER          17568 non-null  int64
 12  IFA_FLOW                   17568 non-null  int64
 13  IFA2_FLOW                  17568 non-null  int64
 14  BRITNED_FLOW               17568 

In [7]:
df.isna().sum()

SETTLEMENT_DATE              0
SETTLEMENT_PERIOD            0
ND                           0
TSD                          0
ENGLAND_WALES_DEMAND         0
EMBEDDED_WIND_GENERATION     0
EMBEDDED_WIND_CAPACITY       0
EMBEDDED_SOLAR_GENERATION    0
EMBEDDED_SOLAR_CAPACITY      0
NON_BM_STOR                  0
PUMP_STORAGE_PUMPING         0
SCOTTISH_TRANSFER            0
IFA_FLOW                     0
IFA2_FLOW                    0
BRITNED_FLOW                 0
MOYLE_FLOW                   0
EAST_WEST_FLOW               0
NEMO_FLOW                    0
NSL_FLOW                     0
ELECLINK_FLOW                0
VIKING_FLOW                  0
GREENLINK_FLOW               0
dtype: int64

## 3. Data Cleaning

Before conducting the analysis, the dataset was inspected for data quality issues, including missing values, duplicate observations, incorrect data types, and inconsistent values.

In [8]:
df.dtypes

SETTLEMENT_DATE                str
SETTLEMENT_PERIOD            int64
ND                           int64
TSD                          int64
ENGLAND_WALES_DEMAND         int64
EMBEDDED_WIND_GENERATION     int64
EMBEDDED_WIND_CAPACITY       int64
EMBEDDED_SOLAR_GENERATION    int64
EMBEDDED_SOLAR_CAPACITY      int64
NON_BM_STOR                  int64
PUMP_STORAGE_PUMPING         int64
SCOTTISH_TRANSFER            int64
IFA_FLOW                     int64
IFA2_FLOW                    int64
BRITNED_FLOW                 int64
MOYLE_FLOW                   int64
EAST_WEST_FLOW               int64
NEMO_FLOW                    int64
NSL_FLOW                     int64
ELECLINK_FLOW                int64
VIKING_FLOW                  int64
GREENLINK_FLOW               int64
dtype: object

In [9]:
df["SETTLEMENT_DATE"] = pd.to_datetime(
    df["SETTLEMENT_DATE"],
    format="%d-%b-%Y"
)

In [10]:
df["SETTLEMENT_DATE"].dtype

dtype('<M8[us]')

In [11]:
df[["SETTLEMENT_DATE", "SETTLEMENT_PERIOD"]].head()

,SETTLEMENT_DATE,SETTLEMENT_PERIOD
0,2024-01-01,1
1,2024-01-01,2
2,2024-01-01,3
3,2024-01-01,4
4,2024-01-01,5


In [12]:
df.duplicated()

0        False
1        False
2        False
3        False
4        False
         ...  
17563    False
17564    False
17565    False
17566    False
17567    False
Length: 17568, dtype: bool

In [13]:
df.duplicated().sum()

np.int64(0)

In [14]:
df.isna().sum()

SETTLEMENT_DATE              0
SETTLEMENT_PERIOD            0
ND                           0
TSD                          0
ENGLAND_WALES_DEMAND         0
EMBEDDED_WIND_GENERATION     0
EMBEDDED_WIND_CAPACITY       0
EMBEDDED_SOLAR_GENERATION    0
EMBEDDED_SOLAR_CAPACITY      0
NON_BM_STOR                  0
PUMP_STORAGE_PUMPING         0
SCOTTISH_TRANSFER            0
IFA_FLOW                     0
IFA2_FLOW                    0
BRITNED_FLOW                 0
MOYLE_FLOW                   0
EAST_WEST_FLOW               0
NEMO_FLOW                    0
NSL_FLOW                     0
ELECLINK_FLOW                0
VIKING_FLOW                  0
GREENLINK_FLOW               0
dtype: int64

In [15]:
df.describe()

,SETTLEMENT_DATE,SETTLEMENT_PERIOD,ND,TSD,ENGLAND_WALES_DEMAND,EMBEDDED_WIND_GENERATION,EMBEDDED_WIND_CAPACITY,EMBEDDED_SOLAR_GENERATION,EMBEDDED_SOLAR_CAPACITY,NON_BM_STOR,...,IFA_FLOW,IFA2_FLOW,BRITNED_FLOW,MOYLE_FLOW,EAST_WEST_FLOW,NEMO_FLOW,NSL_FLOW,ELECLINK_FLOW,VIKING_FLOW,GREENLINK_FLOW
count,17568,17568.000000,17568.000000,17568.000000,17568.000000,17568.000000,17568.000000,17568.000000,17568.000000,17568.0,...,17568.000000,17568.000000,17568.000000,17568.000000,17568.000000,17568.000000,17568.000000,17568.000000,17568.000000,17568.000000
mean,2024-07-01 12:34:25.573770,24.500228,26286.762978,28232.834358,24161.055726,1932.204918,6578.217782,1588.168545,17783.524989,0.0,...,1150.760303,472.830829,180.481728,-284.268841,-305.359460,473.832423,1095.293317,590.681808,416.889230,-0.064891
min,2024-01-01 00:00:00,1.000000,15071.000000,17138.000000,13740.000000,229.000000,6488.000000,0.000000,16793.000000,0.0,...,-2070.000000,-1029.000000,-1079.000000,-453.000000,-532.000000,-1024.000000,-1453.000000,-1024.000000,-1456.000000,-159.000000
25%,2024-04-01 00:00:00,12.750000,21574.750000,23799.500000,19847.750000,922.000000,6562.000000,0.000000,17203.000000,0.0,...,963.000000,-1.000000,-211.000000,-451.000000,-528.000000,78.000000,997.000000,0.000000,-24.250000,0.000000
50%,2024-07-02 00:00:00,24.500000,25361.000000,27079.000000,23259.500000,1601.000000,6562.000000,3.000000,17815.000000,0.0,...,1109.000000,489.000000,9.500000,-380.000000,-431.000000,702.000000,1397.000000,944.000000,604.000000,0.000000
75%,2024-10-01 00:00:00,36.250000,30207.000000,31893.250000,27769.250000,2687.000000,6622.000000,2542.000000,18457.000000,0.0,...,1860.000000,991.000000,854.000000,-227.000000,-30.000000,999.000000,1398.000000,996.000000,989.000000,0.000000
max,2024-12-31 00:00:00,50.000000,45202.000000,47507.000000,41130.000000,5962.000000,6622.000000,11498.000000,18720.000000,0.0,...,2019.000000,993.000000,1066.000000,400.000000,504.000000,1020.000000,1400.000000,1001.000000,1436.000000,27.000000
std,NaN,13.854188,5961.155946,5746.741070,5444.530688,1266.052754,39.002274,2468.029143,623.078523,0.0,...,785.078129,488.463957,659.923018,229.971585,256.698371,593.716791,573.890998,478.747386,758.793645,1.449831


In [16]:
df.loc[df["ND"].idxmax()]

SETTLEMENT_DATE              2024-01-15 00:00:00
SETTLEMENT_PERIOD                             36
ND                                         45202
TSD                                        46976
ENGLAND_WALES_DEMAND                       41074
EMBEDDED_WIND_GENERATION                    1562
EMBEDDED_WIND_CAPACITY                      6488
EMBEDDED_SOLAR_GENERATION                      0
EMBEDDED_SOLAR_CAPACITY                    16823
NON_BM_STOR                                    0
PUMP_STORAGE_PUMPING                           9
SCOTTISH_TRANSFER                           1377
IFA_FLOW                                    -178
IFA2_FLOW                                     -4
BRITNED_FLOW                                 941
MOYLE_FLOW                                  -450
EAST_WEST_FLOW                              -530
NEMO_FLOW                                      0
NSL_FLOW                                     451
ELECLINK_FLOW                                 -3
VIKING_FLOW         

## 4. Feature Engineering

Additional time-based features will be created from the settlement date and settlement period to support analysis of daily, weekly, monthly, and seasonal electricity demand patterns.

In [17]:
df["DATETIME"] = (
    df["SETTLEMENT_DATE"]
    + pd.to_timedelta((df["SETTLEMENT_PERIOD"] - 1) * 30, unit="m")
)

In [18]:
df[["SETTLEMENT_DATE", "SETTLEMENT_PERIOD", "DATETIME", "ND"]].head()

,SETTLEMENT_DATE,SETTLEMENT_PERIOD,DATETIME,ND
0,2024-01-01,1,2024-01-01 00:00:00,21783
1,2024-01-01,2,2024-01-01 00:30:00,22521
2,2024-01-01,3,2024-01-01 01:00:00,22194
3,2024-01-01,4,2024-01-01 01:30:00,21510
4,2024-01-01,5,2024-01-01 02:00:00,20619


### Settlement Period Validation

A standard day contains 48 half-hour settlement periods. The dataset contains settlement periods greater than 48, so these observations are investigated before further time-based feature engineering.

In [19]:
df["SETTLEMENT_PERIOD"].value_counts().sort_index()

SETTLEMENT_PERIOD
1     366
2     366
3     366
4     366
5     366
6     366
7     366
8     366
9     366
10    366
11    366
12    366
13    366
14    366
15    366
16    366
17    366
18    366
19    366
20    366
21    366
22    366
23    366
24    366
25    366
26    366
27    366
28    366
29    366
30    366
31    366
32    366
33    366
34    366
35    366
36    366
37    366
38    366
39    366
40    366
41    366
42    366
43    366
44    366
45    366
46    366
47    365
48    365
49      1
50      1
Name: count, dtype: int64

In [20]:
df[df["SETTLEMENT_PERIOD"] > 48][
    ["SETTLEMENT_DATE", "SETTLEMENT_PERIOD", "ND"]
]

,SETTLEMENT_DATE,SETTLEMENT_PERIOD,ND
14446,2024-10-27,49,21183
14447,2024-10-27,50,20226


In [21]:
df.groupby("SETTLEMENT_DATE").size().sort_values().head()

SETTLEMENT_DATE
2024-03-31    46
2024-09-05    48
2024-09-04    48
2024-09-03    48
2024-09-02    48
dtype: int64

**Finding:** Most days contain 48 half-hour settlement periods. However, 31 March 2024 contains 46 periods and 27 October 2024 contains 50 periods. These dates correspond to the UK daylight-saving clock changes, when the clocks move forward and backward respectively.

In [22]:
df[df["SETTLEMENT_PERIOD"] > 48][
    ["SETTLEMENT_DATE", "SETTLEMENT_PERIOD", "DATETIME", "ND"]
]

,SETTLEMENT_DATE,SETTLEMENT_PERIOD,DATETIME,ND
14446,2024-10-27,49,2024-10-28 00:00:00,21183
14447,2024-10-27,50,2024-10-28 00:30:00,20226


In [23]:
df["DAY_OF_WEEK"] = df["SETTLEMENT_DATE"].dt.day_name()

In [24]:
df[["SETTLEMENT_DATE", "DAY_OF_WEEK"]].head()

,SETTLEMENT_DATE,DAY_OF_WEEK
0,2024-01-01,Monday
1,2024-01-01,Monday
2,2024-01-01,Monday
3,2024-01-01,Monday
4,2024-01-01,Monday


In [25]:
df["IS_WEEKEND"] = df["SETTLEMENT_DATE"].dt.dayofweek >= 5

In [26]:
df[
    ["SETTLEMENT_DATE", "DAY_OF_WEEK", "IS_WEEKEND"]
].head()

,SETTLEMENT_DATE,DAY_OF_WEEK,IS_WEEKEND
0,2024-01-01,Monday,False
1,2024-01-01,Monday,False
2,2024-01-01,Monday,False
3,2024-01-01,Monday,False
4,2024-01-01,Monday,False


In [27]:
df["MONTH"] = df["SETTLEMENT_DATE"].dt.month_name()

In [28]:
df[["SETTLEMENT_DATE", "MONTH"]].head()

,SETTLEMENT_DATE,MONTH
0,2024-01-01,January
1,2024-01-01,January
2,2024-01-01,January
3,2024-01-01,January
4,2024-01-01,January


In [29]:
df["MONTH_NUM"] = df["SETTLEMENT_DATE"].dt.month

In [30]:
season_map = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn"
}

df["SEASON"] = df["MONTH_NUM"].map(season_map)

In [31]:
df[
    ["SETTLEMENT_DATE", "DAY_OF_WEEK", "IS_WEEKEND",
     "MONTH", "MONTH_NUM", "SEASON"]
].sample(10)

,SETTLEMENT_DATE,DAY_OF_WEEK,IS_WEEKEND,MONTH,MONTH_NUM,SEASON
14438,2024-10-27,Sunday,True,October,10,Autumn
6642,2024-05-18,Saturday,True,May,5,Spring
4685,2024-04-07,Sunday,True,April,4,Spring
14423,2024-10-27,Sunday,True,October,10,Autumn
7857,2024-06-12,Wednesday,False,June,6,Summer
2062,2024-02-12,Monday,False,February,2,Winter
14969,2024-11-07,Thursday,False,November,11,Autumn
7763,2024-06-10,Monday,False,June,6,Summer
1527,2024-02-01,Thursday,False,February,2,Winter
16521,2024-12-10,Tuesday,False,December,12,Winter
